# 01 — The systems: generator, program, grounding check, agent, cascade

Shows each piece of Block 3 on small examples and on real evaluation questions. Everything is
done by `ragbench`; no Ollama server or GPU is needed: the LLM outputs are read from the result
files written by `make b3`.

In [1]:
import json
from pathlib import Path

from ragbench import b3
from ragbench import generation as G
from ragbench import program as P

R = Path("../results/evaluation")


def rows(system):
    return {r["qid"]: r for r in map(json.loads, (R / f"{system}.jsonl").open())}


fixed, agentic, cascade = rows("fixed"), rows("agentic"), rows("cascade")

## 1. The generator's instructions

The same prompt (version 4, frozen on the development questions) is used by every system. Note
what it forbids: arithmetic, unit conversion, nesting, and guessing.

In [2]:
print(G.SYSTEM_PROMPT)

You answer questions about financial reports using ONLY the context chunks you are given. You never compute the answer yourself: you write the steps of a calculation.

Steps:
- Each step is one operation on exactly two arguments: add, subtract, multiply, divide,
  exp (arg1 to the power arg2), greater ("yes" if arg1 > arg2 else "no").
- An argument is one of:
  * a number copied from a chunk EXACTLY as written there, only without $, commas and %
    ("$ 161.9 million" -> 161.9: never convert units), and in argN_chunk the id of the
    <chunk> tag it was copied from. A copied number never has argN_chunk 0;
  * #n, the result of an EARLIER step: #0 is the first step, #1 the second, and so on;
  * a constant: const_1 const_2 const_3 const_4 const_5 const_6 const_7 const_9 const_10
    const_100 const_1000 const_1000000 const_m1 (= -1).
  For #n and constants, argN_chunk is 0.
- A percentage or a growth rate stops at the division: do not multiply by 100.
- Rarely needed: table_sum, table_a

## 2. What the generator sees

Chunks are tagged with ids 1..n, and the model cites these ids in each argument.

In [3]:
chunks = [
    "Apple Inc. 2017 annual report, revenue table. Revenue: 2016 $ 5,735 million; 2017 $ 5,829 million.",
    "Apple Inc. 2017 annual report. Operating expenses grew 3 % in 2017.",
]
messages = G.answer_messages("Apple: what was the percentage change in revenue from 2016 to 2017?", chunks)
print(messages[1]["content"])

Context (2 chunks, ids 1 to 2):
<chunk id="1">
Apple Inc. 2017 annual report, revenue table. Revenue: 2016 $ 5,735 million; 2017 $ 5,829 million.
</chunk>
<chunk id="2">
Apple Inc. 2017 annual report. Operating expenses grew 3 % in 2017.
</chunk>

Question: Apple: what was the percentage change in revenue from 2016 to 2017?


## 3. From JSON steps to a program, executed by `ragbench`

The model returns structured steps; `ragbench` assembles the FinQA program and executes it. The
LLM never computes. The match rule accepts 0.5 % of relative error, and a percentage and its
fraction (1.64 vs 0.0164) both count.

In [4]:
output = json.dumps({"status": "answer", "steps": [
    {"op": "subtract", "arg1": "5829", "arg1_chunk": 1, "arg2": "5735", "arg2_chunk": 1},
    {"op": "divide", "arg1": "#0", "arg1_chunk": 0, "arg2": "5735", "arg2_chunk": 1},
]})
answer = G.parse_answer(output)
value = P.execute(answer.program)
print("program :", answer.program)
print("operands:", answer.operands)
print("value   :", value)
print("matches 0.01639 ?", P.answers_match(value, "0.01639"), "| matches 1.64 % ?", P.answers_match(value, "1.64%"))

program : subtract(5829, 5735), divide(#0, 5735)
operands: [('5829', 1), ('5735', 1), ('5735', 1)]
value   : 0.016390584132519617
matches 0.01639 ? True | matches 1.64 % ? True


## 4. The grounding check

Grounded = every number of the program appears in the chunk it cites. Three cases: correct
citation, wrong citation (the number exists, but in another chunk), and a program with no number
at all, which is **vacuously** grounded.

In [5]:
def steps(*s):
    return G.parse_answer(json.dumps({"status": "answer", "steps": list(s)}))


good = answer
wrong_citation = steps({"op": "subtract", "arg1": "5829", "arg1_chunk": 2, "arg2": "5735", "arg2_chunk": 1})
no_numbers = steps({"op": "divide", "arg1": "#0", "arg1_chunk": 0, "arg2": "#1", "arg2_chunk": 0})
for name, a in [("good", good), ("wrong citation", wrong_citation), ("no numbers", no_numbers)]:
    print(f"{name:15s} grounded={G.is_grounded(a, chunks)}  program={a.program}")

good            grounded=True  program=subtract(5829, 5735), divide(#0, 5735)
wrong citation  grounded=False  program=subtract(5829, 5735)
no numbers      grounded=True  program=divide(#0, #1)


The third case happened in the evaluation: fixed wrote `divide(#0, #1)` for a question
where every gold chunk was in context. Grounded, but invalid (there is no step #0 to refer to).

In [6]:
r = fixed["AWK/2014/page_121.pdf-4"]
{k: r[k] for k in ("pass_all_ctx", "program", "program_valid", "error", "grounded", "gold", "correct")}

{'pass_all_ctx': 1.0,
 'program': 'divide(#0, #1)',
 'program_valid': False,
 'error': 'bad reference #0',
 'grounded': True,
 'gold': '0.01197',
 'correct': False}

## 5. The agent at work: a question it loses

Fixed answered this question correctly. The agent started from the same top-10, then spent its
six tool calls; only the last one added evidence. Budget exhausted → `insufficient_context`, by
the frozen rule.

In [7]:
qid = "ABMD/2008/page_86.pdf-4"
print("fixed  :", fixed[qid]["status"], "correct =", fixed[qid]["correct"], "| program:", fixed[qid]["program"])
print("agentic:", agentic[qid]["status"], "| stop:", agentic[qid]["stop"], "| tool calls:", agentic[qid]["tool_calls"])
for n, step in enumerate(agentic[qid]["trace"], start=1):
    print(f"  {n}. {step['tool']:16s} {json.dumps(step['args'])[:70]:72s} new chunks: {len(step['added'])}")

fixed  : answer correct = True | program: add(5.6, 5.583)
agentic: insufficient_context | stop: tool_budget | tool calls: 6
  1. keyword_search   {"query": "2019 Impella contingent payments FDA approval millions"}      new chunks: 0
  2. chunk_read       {"chunk_id": "c158"}                                                     new chunks: 0
  3. chunk_read       {"chunk_id": "c157"}                                                     new chunks: 0
  4. semantic_search  {"query": "2019 Abiomed Impella contingent payments FDA approval"}       new chunks: 0
  5. chunk_read       {"chunk_id": "c159"}                                                     new chunks: 0
  6. page_read        {"chunk_id": "c158"}                                                     new chunks: 2


## 6. The agent at work: a question it wins

Here fixed again wrote an invalid chain (`divide(#0, #1)`). The agent added chunks, and on that
longer context the generator wrote a valid one-step program.

In [8]:
qid = "ETFC/2011/page_144.pdf-1"
for name, r in [("fixed", fixed[qid]), ("agentic", agentic[qid])]:
    print(f"{name:8s} program={r['program']!r:45s} value={r['value']} correct={r['correct']}")
print("gold:", fixed[qid]["gold"], "| chunks added by the agent:", agentic[qid]["added_chunks"])

fixed    program='divide(#0, #1)'                              value=None correct=False
agentic  program='divide(110000, 65000)'                       value=1.6923076923076923 correct=True
gold: 1.69231 | chunks added by the agent: 3


### Pitfall: a loose match on small answers

The frozen match rule is `|pred - gold| <= 0.005 * max(|gold|, 1)`. For an answer below 1 (most
FinQA ratios and growth rates) the tolerance is an **absolute** 0.005, so a few percent of
relative error still counts as correct. Example from the evaluation: the agent's answer is 2.7 %
off the gold value and is scored correct. The rule applies to every system alike.

In [9]:
r = agentic["AWK/2014/page_121.pdf-4"]
pred, gold = r["value"] / 100, P.to_number(r["gold"])
print(f"pred {pred:.5f}  gold {gold:.5f}  relative error {abs(pred - gold) / gold:.1%}  scored correct: {r['correct']}")
print("tolerance used:", 0.005 * max(abs(gold), 1))

pred 0.01165  gold 0.01197  relative error 2.7%  scored correct: True
tolerance used: 0.005


## 7. The cascade's routing

The three signals of the protocol, as implemented in `ragbench.b3.escalation_reasons`, with the
frozen threshold.

In [10]:
threshold = b3.load_threshold()
print("threshold:", round(threshold, 4))
for case in [
    {"status": "answer", "grounded": True},
    {"status": "answer", "grounded": False},
    {"status": "insufficient_context", "grounded": False},
]:
    for top1 in (0.95, 0.5):
        print(case, "top-1", top1, "->", b3.escalation_reasons(case, top1, threshold) or "keep fixed")

threshold: 0.8886
{'status': 'answer', 'grounded': True} top-1 0.95 -> keep fixed
{'status': 'answer', 'grounded': True} top-1 0.5 -> ['low_score']
{'status': 'answer', 'grounded': False} top-1 0.95 -> ['ungrounded']
{'status': 'answer', 'grounded': False} top-1 0.5 -> ['ungrounded', 'low_score']
{'status': 'insufficient_context', 'grounded': False} top-1 0.95 -> ['refusal']
{'status': 'insufficient_context', 'grounded': False} top-1 0.5 -> ['refusal', 'low_score']


In [11]:
from collections import Counter

Counter(tuple(r["escalation_reasons"]) or ("kept fixed",) for r in cascade.values())

Counter({('ungrounded',): 43,
         ('kept fixed',): 35,
         ('low_score',): 10,
         ('ungrounded', 'low_score'): 7,
         ('refusal', 'low_score'): 3,
         ('refusal',): 2})